In [11]:
!python3 -m pip install torch torchvision opencv-python pillow matplotlib tqdm seaborn

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 23.0.1 -> 25.2
[notice] To update, run: python3 -m pip install --upgrade pip


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from PIL import Image
import cv2
import numpy as np
import os
from PIL import ImageFilter 
from tqdm import tqdm
import matplotlib.pyplot as plt
from torchvision import datasets, transforms
from skimage.metrics import structural_similarity as ssim
import numpy as np
from astropy.coordinates import SkyCoord
from astropy.io import fits
import astropy.units as u
import matplotlib.patheffects as path_effects
from PIL import Image
from functools import lru_cache 
from PIL import Image, ImageFilter

In [ ]:
def calculate_normalization_params(all_images):
    """
    calculate normalization params on full dataset
    """
    all_pixels = np.concatenate([img.flatten() for img in all_images])
    
    p5 = np.percentile(all_pixels, 5)  
    p95 = np.percentile(all_pixels, 95) 
    
    gamma = 2
    
    return {
        'p5': p5,
        'p95': p95, 
        'gamma': gamma,
        'range_min': np.min(all_pixels),
        'range_max': np.max(all_pixels)
    }

def normalize_image(image, params):
    """
    normalize one image
    """
    p5, p95, gamma = params['p5'], params['p95'], params['gamma']
    
    # cut emissions
    image_clipped = np.clip(image, p5, p95)
    image_scaled = (image_clipped - p5) / (p95 - p5)  
    
    # Nonlinear transformation (gamma 2 from article)
    image_transformed = image_scaled ** (1/gamma)
    image_normalized = 2 * image_transformed - 1
    
    return image_normalized

def denormalize_image(image_normalized, params):
    """
    reverse conversion
    """
    p5, p95, gamma = params['p5'], params['p95'], params['gamma']
    image_transformed = (image_normalized + 1) / 2
    
    # reverse nononlinear transformation 
    image_scaled = image_transformed ** gamma
    image_original = image_scaled * (p95 - p5) + p5
    
    return image_original

In [ ]:
def load_dataset_fits(filename):
    with fits.open(filename) as hdul:
        dataset = {}
        header = hdul[0].header
        metadata = {
            'diameter_arcmin': header.get('DIAMETER', 16),
            'pixscale_arcmin': header.get('PIXSCALE', 0.5),
            'patch_shape': (header.get('SHAPE0', 32), header.get('SHAPE1', 32))
        }
        
        centers_hdu = hdul['CENTERS']
        centers_data = centers_hdu.data
        centers = list(zip(centers_data['RA'], centers_data['DEC']))
        metadata['centers'] = centers
        
        if 'ACT_PATCHES' in hdul and 'PLANCK_PATCHES' in hdul:
            act_cube = hdul['ACT_PATCHES'].data
            planck_cube = hdul['PLANCK_PATCHES'].data
            dataset['ACT'] = [act_cube[i] for i in range(act_cube.shape[0])]
            dataset['Planck'] = [planck_cube[i] for i in range(planck_cube.shape[0])]
        else:
            act_patches = []
            planck_patches = []
            for hdu in hdul:
                if hdu.name.startswith('ACT_'):
                    act_patches.append(hdu.data)
                elif hdu.name.startswith('PLANCK_'):
                    planck_patches.append(hdu.data)
            dataset['ACT'] = act_patches
            dataset['Planck'] = planck_patches
        
        dataset['metadata'] = metadata
        return dataset

In [ ]:
dataset_fits = load_dataset_fits("/home/jupyter/datasphere/filestore/datasets/mask_dataset.fits")

In [ ]:
def calculate_mse(img_true, img_pred):
    return np.mean((img_true - img_pred) ** 2)

def calculate_psnr(img_true, img_pred):
    mse = calculate_mse(img_true, img_pred)
    if mse == 0:
        return float('inf')
    max_pixel = np.max(img_true)
    psnr = 20 * np.log10(max_pixel) - 10 * np.log10(mse)
    return psnr

def calculate_ssim(img_true, img_pred):
    min_dim = min(img_true.shape[0], img_true.shape[1])
    if min_dim >= 11:
        win_size = 11  
    elif min_dim >= 7:
        win_size = 7   
    else:
        return float('nan')
    
    return ssim(img_true, img_pred, 
                win_size=win_size,  
                data_range=img_true.max()-img_true.min(),
                channel_axis=-1 if img_true.ndim == 3 else None,  
                c1=1e-4, c2=9e-4)

In [ ]:
class ImageDataset(Dataset):
    def __init__(self, transform=None):
        self.transform = transform 
        self.dataset_fits = load_dataset_fits("/home/jupyter/datasphere/filestore/datasets/mask_dataset.fits")
    
    def __len__(self):
        return len(self.dataset_fits['metadata']['centers'])
    
    def __getitem__(self, idx):
        centers = self.dataset_fits['metadata']['centers']
        if idx >= len(centers):
            return None
            
        ra, dec = centers[idx]
        act_patch = self.dataset_fits['ACT'][idx]
        array = np.array(act_patch)
        def normalize_to_image(array):
            normalized = (array - array.min()) / (array.max() - array.min())
            image_array = (normalized * 255).astype(np.uint8)
            return Image.fromarray(image_array, mode='L').convert('RGB')

        image = normalize_to_image(array)
        
        original = image
        blurred = image.filter(ImageFilter.GaussianBlur(radius=1))
        
        if self.transform:
            original = self.transform(original)
            blurred = self.transform(blurred)
            
        return blurred, original

In [ ]:
def get_data(index=0):
    centers = dataset_fits['metadata']['centers']
    if index >= len(centers):
        print(f"Index {index} is more than number of centres ({len(centers)})")
        return None
    
    ra, dec = centers[index]
    act_patch = dataset_fits['ACT'][index]
    array = np.array(act_patch)
    
    normalized = (array - array.min()) / (array.max() - array.min())
    return torch.FloatTensor(normalized).unsqueeze(0)  # (1, H, W)

class RawDataDataset(Dataset):
    def __init__(self, transform=None):
        self.transform = transform
    
    def __len__(self):
        return len(dataset_fits['metadata']['centers'])

    def __getitem__(self, idx):
        centers = dataset_fits['metadata']['centers']
        if idx >= len(centers):
            return None
            
        act_patch = dataset_fits['ACT'][idx]
        array = np.array(act_patch)
        
        normalized = (array - array.min()) / (array.max() - array.min())
        data_tensor = torch.FloatTensor(normalized).unsqueeze(0)  # (1, H, W)
        
        original = data_tensor
        
        blur_kernel = torch.ones(1, 1, 3, 3) / 9.0
        blurred = torch.nn.functional.conv2d(original.unsqueeze(0), blur_kernel, padding=1)
        blurred = blurred.squeeze(0)  # (1, H, W)
            
        return blurred, original

class SimpleDiffusionEnhancer(nn.Module):
    
    def __init__(self, in_channels=1, base_channels=64): 
        super().__init__()
        
        self.enc1 = self._make_block(in_channels, base_channels)
        self.enc2 = self._make_block(base_channels, base_channels*2)
        self.enc3 = self._make_block(base_channels*2, base_channels*4)
        
        self.time_embed = nn.Sequential(
            nn.Linear(1, base_channels*4),
            nn.ReLU(),
            nn.Linear(base_channels*4, base_channels*4)
        )
        
        self.bottleneck = self._make_block(base_channels*8, base_channels*4)
        
        self.dec3 = self._make_block(base_channels*8, base_channels*2)
        self.dec2 = self._make_block(base_channels*4, base_channels)
        self.dec1 = nn.Conv2d(base_channels*2, in_channels, 3, padding=1)  
        
    def _make_block(self, in_ch, out_ch):
        return nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.ReLU()
        )
    
    def forward(self, x, t):
        x1 = self.enc1(x)      
        x2 = self.enc2(x1)     
        x3 = self.enc3(x2)     
        
        t_embed = self.time_embed(t.view(-1, 1))
        t_embed = t_embed.view(x3.shape[0], -1, 1, 1)
        t_embed = t_embed.expand(-1, -1, x3.shape[2], x3.shape[3])
        
        b_input = torch.cat([x3, t_embed], dim=1)
        b = self.bottleneck(b_input)
        
        d3 = self.dec3(torch.cat([b, x3], dim=1))
        d2 = self.dec2(torch.cat([d3, x2], dim=1))
        d1 = self.dec1(torch.cat([d2, x1], dim=1))
        
        return d1

class DiffusionProcess:
    def __init__(self, beta_start=1e-4, beta_end=0.02, num_timesteps=1000, device='cuda'):
        self.device = device
        self.num_timesteps = num_timesteps
        self.betas = torch.linspace(beta_start, beta_end, num_timesteps, device=device)
        self.alphas = 1. - self.betas
        self.alpha_bars = torch.cumprod(self.alphas, dim=0)

    def add_noise(self, x0, t):
        sqrt_alpha_bar = torch.sqrt(self.alpha_bars[t])
        sqrt_one_minus_alpha_bar = torch.sqrt(1. - self.alpha_bars[t])
        
        noise = torch.randn_like(x0)
        x_t = sqrt_alpha_bar[:, None, None, None] * x0 + sqrt_one_minus_alpha_bar[:, None, None, None] * noise
        
        return x_t, noise
    
    def sample_timesteps(self, batch_size):
        return torch.randint(0, self.num_timesteps, (batch_size,), device=self.device)
    
    def sample(self, model, x, num_samples=1, device='cuda', steps=10):
        model.eval()
        x = x.to(device)
        
        with torch.no_grad():
            noisy_img = torch.randn_like(x)
            noisy_img = x
            
            for t in range(steps-1, -1, -1):
                t_batch = torch.tensor([t] * x.shape[0]).to(device)
                
                predicted_noise = model(noisy_img, t_batch.float() / self.num_timesteps)
                
                alpha = self.alphas[t].view(-1, 1, 1, 1)
                alpha_bar = self.alpha_bars[t].view(-1, 1, 1, 1)
                beta = self.betas[t].view(-1, 1, 1, 1)
                
                if t > 0:
                    noise = torch.randn_like(noisy_img)
                else:
                    noise = 0
                
                noisy_img = (1 / torch.sqrt(alpha)) * (
                    noisy_img - (beta / torch.sqrt(1 - alpha_bar)) * predicted_noise
                ) + torch.sqrt(beta) * noise
        
        return noisy_img

def train_model():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    batch_size = 32
    epochs = 25
    learning_rate = 0.0008
    
    dataset = RawDataDataset()
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    
    model = SimpleDiffusionEnhancer(in_channels=1).to(device)  
    diffusion = DiffusionProcess(device=device)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    criterion = nn.MSELoss()
    
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        epoch_metrics = {'mse': 0.0, 'psnr': 0.0, 'ssim': 0.0}
        
        for low_res, high_res in tqdm(dataloader):
            low_res = low_res.to(device)
            high_res = high_res.to(device)
            
            t = diffusion.sample_timesteps(high_res.shape[0]).to(device)
            noisy_imgs, noise = diffusion.add_noise(high_res, t)
            
            optimizer.zero_grad()
            outputs = model(noisy_imgs, t.float() / diffusion.num_timesteps)
            loss = criterion(outputs, noise)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
            
            with torch.no_grad():
                alpha_bar = diffusion.alpha_bars[t]
                alpha_bar = alpha_bar[:, None, None, None]
                
                denoised = (noisy_imgs - torch.sqrt(1 - alpha_bar) * outputs) / torch.sqrt(alpha_bar)
                
                pred_np = denoised.cpu().numpy()[:, 0, :, :]  # (B, H, W)
                true_np = high_res.cpu().numpy()[:, 0, :, :]  # (B, H, W)
                
                batch_size = pred_np.shape[0]
                for i in range(batch_size):
                    epoch_metrics['mse'] += calculate_mse(true_np[i], pred_np[i])
                    epoch_metrics['psnr'] += calculate_psnr(true_np[i], pred_np[i])
                    epoch_metrics['ssim'] += calculate_ssim(true_np[i], pred_np[i])
        
        total_images = len(dataloader) * batch_size
        avg_loss = running_loss / len(dataloader)
        avg_mse = epoch_metrics['mse'] / total_images
        avg_psnr = epoch_metrics['psnr'] / total_images  
        avg_ssim = epoch_metrics['ssim'] / total_images
        
        print(f'Epoch [{epoch+1}/{epochs}], Loss: {avg_loss:.6f}, '
              f'MSE: {avg_mse:.4f}, PSNR: {avg_psnr:.2f}, SSIM: {avg_ssim:.4f}')
    
    torch.save(model.state_dict(), 'diffusion_enhancer.pth')
    return model

def normalize_image(tensor, params=None):
    """Normalize tensors"""
    if params is None:
        min_val = tensor.min()
        max_val = tensor.max()
    else:
        min_val, max_val = params
        
    normalized = (tensor - min_val) / (max_val - min_val)
    return normalized

def calculate_normalization_params(tensor):
    """Calculate params"""
    return tensor.min(), tensor.max()


def test_model_loop(model, dataset, num_tests=100):
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model.to(device)
    model.eval()
    
    diffusion = DiffusionProcess(device=device)
    
    for index in range(min(num_tests, len(dataset))):
        with torch.no_grad():
            test_lr, test_hr = dataset[index]
            
            test_lr = test_lr.unsqueeze(0).to(device).float()  # (1, 1, H, W)
            test_hr = test_hr.unsqueeze(0).to(device).float()  # (1, 1, H, W)
            
            generated_hr = diffusion.sample(model, test_lr, 1, device, steps=1).cpu().numpy()
            test_lr = test_lr.cpu().numpy()
            test_hr = test_hr.cpu().numpy()
            
            norm_params = calculate_normalization_params(generated_hr)
            generated_hr = normalize_image(generated_hr, norm_params)
            center = dataset_fits['metadata']['centers'][index]

            plt.figure(figsize=(15, 5))
            m1 = calculate_mse(test_hr[0, 0], generated_hr[0, 0])
            psnr1 = calculate_psnr(test_hr[0, 0], generated_hr[0, 0])
            ssim1 = calculate_ssim(test_hr[0, 0], generated_hr[0, 0])
            
            m2 = calculate_mse(test_hr[0, 0], test_lr[0, 0])
            psnr2 = calculate_psnr(test_hr[0, 0], test_lr[0, 0])
            ssim2 = calculate_ssim(test_hr[0, 0], test_lr[0, 0])

            plt.suptitle(f'Координаты: {center[0]}, {center[1]}', fontsize=16, fontweight='bold', y=0.98)
            plt.subplot(1, 3, 1)
            plt.imshow(test_lr[0, 0], cmap='viridis', vmin=0, vmax=1)
            plt.title('LR Input')
            plt.colorbar()

            plt.subplot(1, 3, 2)
            plt.imshow(generated_hr[0, 0], cmap='viridis', vmin=0, vmax=1)
            plt.title('Generated HR')
            plt.colorbar()

            plt.subplot(1, 3, 3)
            plt.imshow(test_hr[0, 0], cmap='viridis', vmin=0, vmax=1)
            plt.title('True HR')
            plt.colorbar()

            plt.figtext(0.5, 0.08, f'Input image MSE={m2:.4f}, PSNR={psnr2:.4f}, SSIM={ssim2:.4f}', ha='center', fontsize=12, fontweight='bold')
            plt.figtext(0.5, 0.04, f'impoved image MSE={m1:.4f}, PSNR={psnr1:.4f}, SSIM={ssim1:.4f}', ha='center', fontsize=12, fontweight='bold')

            plt.tight_layout(rect=[0, 0.12, 1, 0.95])
            plt.savefig(f'results_elya/new7_results_test_{index}.png')
            
            plt.close()
            
        print(f'end {index}')
    
    return generated_hr


if __name__ == "__main__":
    # train model or load pretrained
    # model = train_model()
    dataset = RawDataDataset()

    model = SimpleDiffusionEnhancer(in_channels=1)
    model.load_state_dict(torch.load('diffusion_enhancer.pth', map_location=torch.device('cpu')))
    
    print("Run")
    test_model_loop(model, dataset, num_tests=100)

Run
end 0
end 1
end 2
end 3
end 4
end 5
end 6
end 7
end 8
end 9
end 10
end 11
end 12
end 13
end 14
end 15
end 16
end 17
end 18
end 19
end 20
end 21
end 22
end 23
end 24
end 25
end 26
end 27
end 28


KeyboardInterrupt: 